# FreshMart Day 1 — Practice: from messy files to a script that runs itself

**The story:** FreshMart has 3 branches — Ikeja, Wuse and Garki. Every week each branch manager sends a sales file.
The files are messy, and someone spends hours fixing them by hand. You will teach Python to do it in 2 seconds.

**You will:**
1. Look at the messy files with Python
2. Fix 6 problems, one at a time
3. Build a sales report automatically
4. Turn your work into a script that runs **by itself** on a timer

**Before you start:** finish Part 1 of `STUDENT_GUIDE.pdf`, and open this notebook from Anaconda Prompt inside this folder.

**How to work:** run each cell with **Shift + Enter**. Where you see `___`, replace it with the right code.
Stuck? The answers are at the end of the guide. Try first!

## Step 0 — Get ready

In [ ]:
import pandas as pd
import glob

pd.set_option("display.max_rows", 100)
print("Ready!")

## Step 1 — Look at one file
First, open `data/raw/ikeja_week41.xlsx` in Excel and look at it. Then read it with Python.

In [ ]:
# Read the Ikeja file into a variable called ikeja, then show the first 10 rows
ikeja = pd.read_excel(___)
ikeja.head(___)

**Ask Python what it sees.** How many rows? Which columns have missing values?

In [ ]:
# Show the column types and how many values each column has
ikeja.___()

In [ ]:
# Count each spelling in the Branch column. How many ways is "Ikeja" written?
ikeja["Branch"].___()

**Write down what you notice.** You should find: different spellings of Ikeja, dates written in different ways,
prices with ₦ signs and commas, an empty quantity, and a row that appears twice.

## Step 2 — Read ALL the files at once
`glob` finds every file that matches a pattern. `*` means "anything".

In [ ]:
# Find every .xlsx file inside data/raw
files = sorted(glob.glob(___))
files

In [ ]:
# Read each file and put them in a list, then stack them into one table
frames = []
for f in files:
    df = pd.read_excel(___)
    frames.append(___)

sales = pd.concat(frames, ignore_index=True)
sales.columns.tolist()

😬 **Problem 1: the columns don't match.** Ikeja says `Qty`, Wuse says `Quantity`, Garki says `quantity ` (with a space).
Python treats them as 3 different columns.

## Step 3 — Fix Problem 1: column names
Two moves: (a) make every name lowercase with no extra spaces, (b) rename the odd ones.

In [ ]:
column_names = {
    "date_sold": "date",
    "qty": "quantity",
    "price": "unit_price",
    "payment_method": ___,
}

frames = []
for f in files:
    df = pd.read_excel(f)
    # strip spaces, make lowercase, replace " " with "_"
    df.columns = df.columns.str.strip().str.___().str.replace(" ", "_")
    df = df.rename(columns=___)
    frames.append(df)

sales = pd.concat(frames, ignore_index=True)
print(sales.columns.tolist())
print(sales.shape)

✅ You should now see exactly 6 columns: `date, branch, product, quantity, unit_price, payment` and 75 rows.

## Step 4 — Fix Problem 2: branch names

In [ ]:
sales["branch"].value_counts()

In [ ]:
sales["branch"] = sales["branch"].str.strip().str.___()      # remove spaces, Make Each Word Capital
sales["branch"] = sales["branch"].str.split().str[___]       # keep only the first word
sales["branch"] = sales["branch"].replace({"Ikj": ___})      # fix the typo
sales["branch"].value_counts()

✅ You should see only **Garki, Ikeja, Wuse** (25, 26, 24 rows).

## Step 5 — Fix Problem 3: dates

In [ ]:
sales["date"].head(10)

In [ ]:
# format="mixed" lets pandas read different styles; dayfirst=True because we write day/month/year
sales["date"] = pd.to_datetime(sales["date"], format=___, dayfirst=___)
sales["date"].value_counts().sort_index()

✅ You should see 7 dates, from **2026-10-03 to 2026-10-09**.

## Step 6 — Fix Problem 4: prices stored as text

In [ ]:
sales["unit_price"].head(10)

In [ ]:
# keep only digits: [^0-9.] means "anything that is NOT a number or a dot"
sales["unit_price"] = sales["unit_price"].astype(str).str.replace(r"[^0-9.]", "", regex=True)
sales["unit_price"] = pd.___(sales["unit_price"])   # turn text into numbers
sales["unit_price"].describe()

✅ `describe()` now works and shows a **min of 350** and **max of 9500**. It only works on numbers!

## Step 7 — Fix Problem 5: duplicate rows

In [ ]:
print("Duplicate rows:", sales.___().sum())
sales = sales.___()
print("Rows left:", len(sales))

✅ 4 duplicates removed, **71 rows** left.

## Step 8 — Fix Problem 6: missing quantities
We can't guess how many were sold. So we **save these rows for the manager to check**, then remove them.

In [ ]:
# Keep only the rows where quantity is empty
missing = sales[sales["quantity"].___()]
missing

In [ ]:
missing.to_excel("output/rows_to_check.xlsx", index=False)
sales = sales.dropna(subset=[___])
sales["quantity"] = sales["quantity"].astype(int)
print("Clean rows:", len(sales))

✅ **66 clean rows.** Open `output/rows_to_check.xlsx` — that's the list we'd email to the managers.

## Step 9 — Build the report

In [ ]:
# revenue = quantity x unit price
sales["revenue"] = ___ * ___

# total revenue for each branch, biggest first
by_branch = sales.groupby(___)["revenue"].sum().sort_values(ascending=False).reset_index()
by_branch

In [ ]:
by_product = (sales.groupby("product")
              .agg(units_sold=("quantity", "sum"), revenue=("revenue", "sum"))
              .sort_values("revenue", ascending=False).reset_index())
by_day = sales.groupby("date")["revenue"].sum().reset_index()
by_payment = sales.groupby("payment")["revenue"].sum().reset_index()
by_product

✅ Ikeja should be top with **₦757,400**. Rice (5kg) is the best-selling product by revenue.

## Step 10 — Save the report to Excel, with today's date in the name

In [ ]:
from datetime import date

today = date.today().isoformat()
report_file = f"output/freshmart_report_{___}.xlsx"

with pd.ExcelWriter(report_file) as writer:
    sales.to_excel(writer, sheet_name="clean_data", index=False)
    by_branch.to_excel(writer, sheet_name="by_branch", index=False)
    # add by_product, by_day and by_payment the same way
    ___

print("Saved", report_file)

🎉 Open the `output` folder and open your report.

---
## Step 11 — From notebook to script
A notebook is for **exploring**. A script is for **running every week without you**.
Open `clean_sales.py` — it is everything we just did, in one file. Then run it:

In [ ]:
!python clean_sales.py

## Step 12 — Make it run by itself
Follow **Day 1, Step 12** in the guide: in a second Anaconda Prompt, inside this folder, type:

```
python scheduler.py
```

Watch it run every minute without you touching anything. Open `output/automation_log.txt` and see the entries pile up.
Press **Ctrl + C** to stop it.

🎉 **Day 1 done.** Next: `Day2_Dashboards.ipynb` (guide, Part 2).